# Model C — 30 Observed Calendar Dates Training

Train on **Jul 13–Aug 11**. This is **30 observed calendar dates spanning 29 elapsed days**. No final out-of-sample test is performed because no post-Aug-11 data exists yet. This model is intended for the final live deployment experiment with Model A and Model B.

In [2]:

import os
import json
import pickle
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from datetime import date
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    accuracy_score,
    confusion_matrix,
)

# Run from the project's notebooks/ directory.
# Project layout:
# ibm_quantum/
#   backup_calibration_history_20260811_1706.csv
#   models/
#   results/
#   notebooks/

PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
CAL_CSV = os.path.join(PROJECT_ROOT, "backup_calibration_history_20260811_1706.csv")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

T1_THRESH = 100.0
T2_THRESH = 50.0
RE_THRESH = 0.05

HIST_FEATURES = [
    "hist_T1_mean", "hist_T1_std", "hist_T1_min", "hist_T1_max",
    "hist_T2_mean", "hist_T2_std", "hist_T2_min", "hist_T2_max",
    "hist_RE_mean", "hist_RE_std", "hist_RE_min", "hist_RE_max",
    "prev_T1", "prev_T2", "prev_RE",
    "hist_coherence_product", "hist_t2_t1_ratio"
]

RF_PARAMS = dict(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

print("Project root:", PROJECT_ROOT)
print("Calibration CSV:", CAL_CSV)

df = pd.read_csv(CAL_CSV)
df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])
for col in ["T1_us", "T2_us", "readout_error"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["qubit"] = pd.to_numeric(df["qubit"], errors="coerce").astype(int)
df = df.dropna(subset=["T1_us", "T2_us", "readout_error"])
df = df.sort_values(["backend", "qubit", "snapshot_date"]).reset_index(drop=True)

print(f"Loaded rows: {len(df):,}")
print(f"Date range: {df['snapshot_date'].min().date()} -> {df['snapshot_date'].max().date()}")
print(f"Unique calendar dates: {df['snapshot_date'].nunique()}")
print(f"Backends: {sorted(df['backend'].unique())}")
print(f"T1 mean: {df['T1_us'].mean():.1f} us")

assert df["T1_us"].mean() <= 10000, "T1 values look like a unit-conversion bug."

df["label"] = (
    (df["T1_us"] > T1_THRESH) &
    (df["T2_us"] > T2_THRESH) &
    (df["readout_error"] < RE_THRESH)
).astype(int)

# Same historical feature logic as Model A.
# Current-session values NEVER enter features.
df["snapshot_id"] = (
    df.groupby(["backend", "qubit"])["snapshot_date"]
      .transform(lambda x: pd.factorize(x)[0])
)

for col, alias in [("T1_us", "T1"), ("T2_us", "T2"), ("readout_error", "RE")]:
    g = df.groupby(["backend", "qubit"])[col]
    df[f"hist_{alias}_mean"] = g.transform(lambda x: x.shift(1).expanding().mean())
    df[f"hist_{alias}_std"]  = g.transform(lambda x: x.shift(1).expanding().std())
    df[f"hist_{alias}_min"]  = g.transform(lambda x: x.shift(1).expanding().min())
    df[f"hist_{alias}_max"]  = g.transform(lambda x: x.shift(1).expanding().max())
    df[f"prev_{alias}"]      = g.transform(lambda x: x.shift(1))

df["hist_coherence_product"] = df["hist_T1_mean"] * df["hist_T2_mean"]
df["hist_t2_t1_ratio"] = df["hist_T2_mean"] / (df["hist_T1_mean"] + 1e-9)

snap0 = df[df["snapshot_id"] == 0]
nan_rates = snap0[HIST_FEATURES].isnull().mean()
assert nan_rates.min() == 1.0, "LEAKAGE DETECTED: snapshot 0 has non-NaN historical features."
print("Leakage audit PASSED — snapshot 0 is 100% NaN")


# ============================================================
# MODEL C — TRAIN JUL 13-AUG 11
# NO OUT-OF-SAMPLE TEST YET
# ============================================================

TRAIN_START, TRAIN_END = "2026-07-13", "2026-08-11"

train = df[
    (df["snapshot_date"] >= TRAIN_START) &
    (df["snapshot_date"] <= TRAIN_END)
].copy()

train = train[train["snapshot_id"] >= 1].dropna(subset=HIST_FEATURES).copy()

observed_dates = train["snapshot_date"].nunique()
elapsed_days = (
    pd.Timestamp(TRAIN_END) - pd.Timestamp(TRAIN_START)
).days

print("\n" + "=" * 65)
print("MODEL C — TRAINING")
print("=" * 65)
print("Training window:", TRAIN_START, "->", TRAIN_END)
print("Observed calendar dates:", observed_dates)
print("Elapsed span:", elapsed_days, "days")
print("Training rows:", len(train))
print("Training viable rate:", round(train["label"].mean(), 4))
print("Backends:", sorted(train["backend"].unique()))

assert len(train) > 0

X_train = train[HIST_FEATURES]
y_train = train["label"]

model_c = RandomForestClassifier(**RF_PARAMS)
model_c.fit(X_train, y_train)

model_c_path = os.path.join(MODELS_DIR, "model_c_30day.pkl")
with open(model_c_path, "wb") as f:
    pickle.dump(model_c, f)

print("\nSaved Model C:", model_c_path)

# Optional sanity check only. NOT a generalization result.
prob = model_c.predict_proba(X_train)[:, 1]
pred = (prob >= 0.5).astype(int)

print("\nIn-sample sanity check ONLY:")
print("AUC:", round(roc_auc_score(y_train, prob), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_train, pred), 4))
print("Do NOT report these as Model C predictive performance.")

metadata = {
    "model_name": "Model_C",
    "model_file": "model_c_30day.pkl",
    "training_start": TRAIN_START,
    "training_end": TRAIN_END,
    "observed_calendar_dates": int(observed_dates),
    "elapsed_span_days": int(elapsed_days),
    "training_rows": int(len(train)),
    "n_features": len(HIST_FEATURES),
    "features": HIST_FEATURES,
    "label_thresholds": {"T1_us": T1_THRESH, "T2_us": T2_THRESH, "RE": RE_THRESH},
    "feature_source": "prior_sessions_only_shift_expand",
    "leakage_status": "NONE_VERIFIED_BY_SNAPSHOT0_AUDIT",
    "rf_params": RF_PARAMS,
    "final_oos_test_status": "PENDING_POST_2026-08-11_DATA",
    "deployment_role": "FINAL_LIVE_EXPERIMENT",
    "trained_on_date": str(date.today()),
    "backends": sorted(train["backend"].unique().tolist())
}

meta_path = os.path.join(RESULTS_DIR, "model_c_30day_metadata.json")
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("Metadata saved:", meta_path)


Project root: D:\Downloads\uday projects\ibm_quantum
Calibration CSV: D:\Downloads\uday projects\ibm_quantum\backup_calibration_history_20260811_1706.csv
Loaded rows: 13,980
Date range: 2026-07-13 -> 2026-08-11
Unique calendar dates: 30
Backends: ['ibm_fez', 'ibm_kingston', 'ibm_marrakesh']
T1 mean: 177.4 us
Leakage audit PASSED — snapshot 0 is 100% NaN

MODEL C — TRAINING
Training window: 2026-07-13 -> 2026-08-11
Observed calendar dates: 28
Elapsed span: 29 days
Training rows: 13048
Training viable rate: 0.6069
Backends: ['ibm_fez', 'ibm_kingston', 'ibm_marrakesh']

Saved Model C: D:\Downloads\uday projects\ibm_quantum\models\model_c_30day.pkl

In-sample sanity check ONLY:
AUC: 0.9352
Balanced Accuracy: 0.8581
Do NOT report these as Model C predictive performance.
Metadata saved: D:\Downloads\uday projects\ibm_quantum\results\model_c_30day_metadata.json
